In [10]:
import nltk
import string
from datasets import load_dataset
import os
import re

In [2]:
# Создание папки для nltk данных, если её нет
nltk_data_dir = os.path.expanduser('../nltk_data')
if not os.path.exists(nltk_data_dir):
    os.makedirs(nltk_data_dir)

# Добавление пути в nltk
nltk.data.path.append(nltk_data_dir)

# Загрузка всех необходимых ресурсов
resources = ['punkt', 'wordnet', 'omw-1.4', 'punkt_tab', 
             'averaged_perceptron_tagger', 'averaged_perceptron_tagger_eng', 'stopwords']

for resource in resources:
    try:
        nltk.download(resource, download_dir=nltk_data_dir, quiet=False)
    except:
        print(f"Ресурс {resource} уже загружен или произошла ошибка")

[nltk_data] Downloading package punkt to ../nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to ../nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to ../nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package punkt_tab to ../nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     ../nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     ../nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package stopwords to ../nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


# 3. Удаление спецслов

In [41]:
def clean_text(text):

    # Приведение к нижнему регистру
    text = text.lower()

    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)


    text = re.sub(r'<.*?>', '', text)


    text = re.sub(r'[{}[\]()<>]', '', text)

    text = re.sub(r'\d+\.\d+\.\d+\.\d+', '', text)  # IP-адреса

    # Удаление email-адресов
    email_pattern = r'[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}'
    text = re.sub(email_pattern, '', text)

    # Удаление путей к файлам
    text = re.sub(r'\S+/\S+/\S+', '', text)
    text = re.sub(r'\S+\.\S+/\S+', '', text)

    # Удаление упоминаний (@username), хештеги
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'#(\w+)', r'\1', text)

    # ОЧИСТКА ОТ КАВЫЧЕК И СПЕЦСИМВОЛОВ
    text = re.sub(r'[`"\']{2,}', ' ', text)  # удвоенные кавычки
    text = re.sub(r'[`"\']', ' ', text)      # одиночные кавычки
    text = re.sub(r'[\[\]{}()<>]', ' ', text)  # скобки

    # Обработка технических терминов
    # Замена "scsi-1" на "scsi1" или оставляем как есть
    text = re.sub(r'(\w+)-(\d+)', r'\1\2', text)  # убираем дефис в терминах
    text = re.sub(r'(\d+)-(\w+)', r'\1\2', text)

    # Удаление текстовых смайликов
    smile_pattern = r'[:;=][\-^]?[)D\(\[\]pP]+'
    text = re.sub(smile_pattern, '', text)
    
    # Удаление повторяющихся знаков пунктуации (!!!, ???, ...)
    text = re.sub(r'([!?.,])\1+', r'\1', text)  # !!! -> !

    text = re.sub(r'-{2,}', ' ', text)
    
    # Удаление телефонных номеров (опционально)
    text = re.sub(r'\+?\d[\d\s\-\(\)]{7,}\d', '', text)

    text = re.sub(r'[^\w\s]', ' ', text) 
    
    # Удаление лишних пробелов
    text = re.sub(r'\s+', ' ', text)
    text = text.strip()
    
    return text

In [42]:
from nltk.corpus import wordnet
from nltk.corpus import stopwords

stop_words = set(stopwords.words("english"))
additional_stops = {
    'would', 'could', 'should', 'might', 'may', 'get', 'go', 'see', 
    'know', 'like', 'want', 'need', 'say', 'think', 'come', 'take',
    'use', 'make', 'well', 'also', 'even', 'many', 'much', 'still',
    'however', 'though', 'although', 'since', 'yet', 'already'
}
stop_words.update(additional_stops)

def get_wordnet_pos(tag):
    if tag.startswith('J'): return wordnet.ADJ
    elif tag.startswith('V'): return wordnet.VERB
    elif tag.startswith('R'): return wordnet.ADV
    else: return wordnet.NOUN

def remove_stopwords(tokens):
    return [token for token in tokens if token not in stop_words]


# Применение лемматизации
lemmatizer = nltk.WordNetLemmatizer()
# Функция предобработки с лемматизацией
def preprocess_with_lemmatization(text):
    # Приведение к нижнему регистру
    text = clean_text(text)
    # Токенизация
    tokens = nltk.word_tokenize(text)
    tagged = nltk.pos_tag(tokens)

    lemmatized_tokens = [lemmatizer.lemmatize(token, get_wordnet_pos(tag)) for token, tag in tagged
                        if token not in string.punctuation]
    

    return remove_stopwords(lemmatized_tokens)

# 4. Фильтрация по частям речи

In [43]:
from nltk.corpus import wordnet

def filter_by_pos(tokens, pos_tags):
    tagged_tokens = nltk.pos_tag(tokens)

    filtered = []
    for word, tag in tagged_tokens:
        if any(tag.startswith(prefix) for prefix in pos_tags):
            filtered.append(word)
    
    return filtered

def task1(text):
    lemmatized_tokens = preprocess_with_lemmatization(text)
    alphabetic_tokens = [token for token in lemmatized_tokens if token.isalpha()]

    nouns_adjectives = filter_by_pos(alphabetic_tokens, ['N', 'J'])

    return nouns_adjectives

def task2(text):
    lemmatized_tokens = preprocess_with_lemmatization(text)
    alphabetic_tokens = [token for token in lemmatized_tokens if token.isalpha()]

    nouns_adjectives = filter_by_pos(alphabetic_tokens, ['N', 'J', 'V'])

    return nouns_adjectives

In [12]:
# Загрузка датасета Emotion (HuggingFace)
dataset = load_dataset("emotion")

In [44]:
examples = dataset["train"][:50]
texts = examples["text"]

for i, text in enumerate(texts):
    print(f"Пример {i+1}")
    print(text)
    print()

ValueError: Column 'train' doesn't exist.

In [45]:
lemmatized_rezult = [preprocess_with_lemmatization(text) for text in texts]

for i in range(len(texts)):
    print(f"Пример {i+1}")
    print(lemmatized_rezult[i])
    print()

Пример 1
['didnt', 'feel', 'humiliate']

Пример 2
['feel', 'hopeless', 'damned', 'hopeful', 'around', 'someone', 'care', 'awake']

Пример 3
['im', 'grab', 'minute', 'post', 'feel', 'greedy', 'wrong']

Пример 4
['ever', 'feel', 'nostalgic', 'fireplace', 'property']

Пример 5
['feel', 'grouchy']

Пример 6
['ive', 'feel', 'little', 'burdened', 'lately', 'wasnt', 'sure']

Пример 7
['ive', 'milligram', 'time', 'recommended', 'amount', 'ive', 'fall', 'asleep', 'lot', 'faster', 'feel', 'funny']

Пример 8
['feel', 'confuse', 'life', 'teenager', 'jade', 'year', 'old', 'man']

Пример 9
['petronas', 'year', 'feel', 'petronas', 'perform', 'huge', 'profit']

Пример 10
['feel', 'romantic']

Пример 11
['feel', 'suffering', 'mean', 'something']

Пример 12
['feel', 'run', 'divine', 'experience', 'expect', 'type', 'spiritual', 'encounter']

Пример 13
['easy', 'time', 'year', 'feel', 'dissatisfied']

Пример 14
['feel', 'low', 'energy', 'thirsty']

Пример 15
['immense', 'sympathy', 'general', 'point', 'po

In [46]:
# фильтрация существительных и прилагательных
for i, text in enumerate(texts):
    print(f'Пример {i+1}')
    print(task1(text))
    print()

Пример 1
['didnt', 'feel', 'humiliate']

Пример 2
['feel', 'hopeless', 'hopeful', 'someone', 'care', 'awake']

Пример 3
['im', 'grab', 'minute', 'post', 'feel', 'greedy', 'wrong']

Пример 4
['nostalgic', 'fireplace', 'property']

Пример 5
['feel', 'grouchy']

Пример 6
['ive', 'feel', 'wasnt', 'sure']

Пример 7
['ive', 'milligram', 'time', 'amount', 'ive', 'fall', 'asleep', 'lot', 'funny']

Пример 8
['feel', 'confuse', 'life', 'teenager', 'year', 'old', 'man']

Пример 9
['petronas', 'year', 'perform', 'huge', 'profit']

Пример 10
['feel', 'romantic']

Пример 11
['feel', 'mean', 'something']

Пример 12
['feel', 'divine', 'experience', 'type', 'spiritual', 'encounter']

Пример 13
['easy', 'time', 'year', 'feel']

Пример 14
['feel', 'low', 'energy', 'thirsty']

Пример 15
['immense', 'sympathy', 'general', 'point', 'possible', 'proto', 'writer', 'find', 'time', 'write', 'corner', 'life', 'sign', 'agent', 'contract', 'precious']

Пример 16
['feel', 'anxiety', 'side']

Пример 17
['didnt', 'em

In [47]:
# фильтрация существительных и прилагательных и глаголов
for i, text in enumerate(texts):
    print(f'Пример {i+1}')
    print(task2(text))
    print()

Пример 1
['didnt', 'feel', 'humiliate']

Пример 2
['feel', 'hopeless', 'damned', 'hopeful', 'someone', 'care', 'awake']

Пример 3
['im', 'grab', 'minute', 'post', 'feel', 'greedy', 'wrong']

Пример 4
['feel', 'nostalgic', 'fireplace', 'property']

Пример 5
['feel', 'grouchy']

Пример 6
['ive', 'feel', 'burdened', 'wasnt', 'sure']

Пример 7
['ive', 'milligram', 'time', 'recommended', 'amount', 'ive', 'fall', 'asleep', 'lot', 'feel', 'funny']

Пример 8
['feel', 'confuse', 'life', 'teenager', 'jade', 'year', 'old', 'man']

Пример 9
['petronas', 'year', 'feel', 'perform', 'huge', 'profit']

Пример 10
['feel', 'romantic']

Пример 11
['feel', 'suffering', 'mean', 'something']

Пример 12
['feel', 'run', 'divine', 'experience', 'expect', 'type', 'spiritual', 'encounter']

Пример 13
['easy', 'time', 'year', 'feel', 'dissatisfied']

Пример 14
['feel', 'low', 'energy', 'thirsty']

Пример 15
['immense', 'sympathy', 'general', 'point', 'possible', 'proto', 'writer', 'try', 'find', 'time', 'write', 

In [21]:
# Загрузка 20 Newsgroups с 4 классами
dataset = load_dataset("SetFit/20_newsgroups", split="train")

categories = [
    "comp.sys.ibm.pc.hardware",
    "comp.sys.mac.hardware",
    "comp.graphics",
    "comp.windows.x"
]

filtered_texts = [
    example["text"]
    for example in dataset
    if example["label_text"] in categories
][:50]

Repo card metadata block was not found. Setting CardData to empty.


In [48]:
for i, text in enumerate(filtered_texts):
    print(f"Пример {i+1}")
    print(text)
    print()

Пример 1
A fair number of brave souls who upgraded their SI clock oscillator have
shared their experiences for this poll. Please send a brief message detailing
your experiences with the procedure. Top speed attained, CPU rated speed,
add on cards and adapters, heat sinks, hour of usage per day, floppy disk
functionality with 800 and 1.4 m floppies are especially requested.

I will be summarizing in the next two days, so please add to the network
knowledge base if you have done the clock upgrade and haven't answered this
poll. Thanks.

Пример 2
well folks, my mac plus finally gave up the ghost this weekend after
starting life as a 512k way back in 1985.  sooo, i'm in the market for a
new machine a bit sooner than i intended to be...

i'm looking into picking up a powerbook 160 or maybe 180 and have a bunch
of questions that (hopefully) somebody can answer:

* does anybody know any dirt on when the next round of powerbook
introductions are expected?  i'd heard the 185c was supposed to ma

In [49]:
cl_text = [clean_text(text) for text in filtered_texts]

for i, text in enumerate(cl_text):
    print(f"Пример {i+1}")
    print(text)
    print()

Пример 1
a fair number of brave souls who upgraded their si clock oscillator have shared their experiences for this poll please send a brief message detailing your experiences with the procedure top speed attained cpu rated speed add on cards and adapters heat sinks hour of usage per day floppy disk functionality with 800 and 1 4 m floppies are especially requested i will be summarizing in the next two days so please add to the network knowledge base if you have done the clock upgrade and haven t answered this poll thanks

Пример 2
well folks my mac plus finally gave up the ghost this weekend after starting life as a 512k way back in 1985 sooo i m in the market for a new machine a bit sooner than i intended to be i m looking into picking up a powerbook 160 or maybe 180 and have a bunch of questions that hopefully somebody can answer does anybody know any dirt on when the next round of powerbook introductions are expected i d heard the 185c was supposed to make an appearence this summer

In [50]:
lemmatized_result = [preprocess_with_lemmatization(text) for text in filtered_texts]

for i in range(len(filtered_texts)):
    print(f"Пример {i+1}:")
    print(f"Lemmatizing: {lemmatized_result[i]}, length: {len(lemmatized_result[i])}")
    print()

Пример 1:
Lemmatizing: ['fair', 'number', 'brave', 'soul', 'upgrade', 'si', 'clock', 'oscillator', 'share', 'experience', 'poll', 'please', 'send', 'brief', 'message', 'detail', 'experience', 'procedure', 'top', 'speed', 'attain', 'cpu', 'rat', 'speed', 'add', 'card', 'adapter', 'heat', 'sink', 'hour', 'usage', 'per', 'day', 'floppy', 'disk', 'functionality', '800', '1', '4', 'floppy', 'especially', 'request', 'summarize', 'next', 'two', 'day', 'please', 'add', 'network', 'knowledge', 'base', 'clock', 'upgrade', 'answer', 'poll', 'thanks'], length: 56

Пример 2:
Lemmatizing: ['folk', 'mac', 'plus', 'finally', 'give', 'ghost', 'weekend', 'start', 'life', '512k', 'way', 'back', '1985', 'sooo', 'market', 'new', 'machine', 'bit', 'sooner', 'intend', 'look', 'pick', 'powerbook', '160', 'maybe', '180', 'bunch', 'question', 'hopefully', 'somebody', 'answer', 'anybody', 'dirt', 'next', 'round', 'powerbook', 'introduction', 'expect', 'hear', '185c', 'suppose', 'appearence', 'summer', 'hear', 'a

In [51]:
# фильтрация существительных и прилагательных
for i, text in enumerate(filtered_texts):
    print(f'Пример {i+1}')
    print(task1(text))
    print()

Пример 1
['fair', 'number', 'upgrade', 'si', 'clock', 'oscillator', 'share', 'experience', 'poll', 'please', 'brief', 'message', 'detail', 'experience', 'procedure', 'top', 'speed', 'cpu', 'rat', 'speed', 'card', 'adapter', 'heat', 'hour', 'usage', 'day', 'floppy', 'disk', 'functionality', 'floppy', 'summarize', 'next', 'day', 'add', 'network', 'knowledge', 'base', 'clock', 'answer', 'poll', 'thanks']

Пример 2
['folk', 'mac', 'ghost', 'weekend', 'life', 'way', 'sooo', 'market', 'new', 'machine', 'bit', 'sooner', 'intend', 'look', 'pick', 'powerbook', 'bunch', 'question', 'somebody', 'anybody', 'dirt', 'next', 'round', 'powerbook', 'introduction', 'hear', 'suppose', 'appearence', 'summer', 'hear', 'access', 'macleak', 'wonder', 'anybody', 'anybody', 'hear', 'rumor', 'price', 'drop', 'powerbook', 'line', 'duo', 'impression', 'display', 'disk', 'good', 'display', 'yea', 'look', 'great', 'store', 'wow', 'good', 'solicit', 'opinion', 'people', 'day', 'day', 'disk', 'size', 'money', 'active

In [52]:
# фильтрация существительных и прилагательных и глаголов
for i, text in enumerate(filtered_texts):
    print(f'Пример {i+1}')
    print(task2(text))
    print()

Пример 1
['fair', 'number', 'brave', 'soul', 'upgrade', 'si', 'clock', 'oscillator', 'share', 'experience', 'poll', 'please', 'send', 'brief', 'message', 'detail', 'experience', 'procedure', 'top', 'speed', 'attain', 'cpu', 'rat', 'speed', 'add', 'card', 'adapter', 'heat', 'sink', 'hour', 'usage', 'day', 'floppy', 'disk', 'functionality', 'floppy', 'request', 'summarize', 'next', 'day', 'please', 'add', 'network', 'knowledge', 'base', 'clock', 'upgrade', 'answer', 'poll', 'thanks']

Пример 2
['folk', 'mac', 'give', 'ghost', 'weekend', 'start', 'life', 'way', 'sooo', 'market', 'new', 'machine', 'bit', 'sooner', 'intend', 'look', 'pick', 'powerbook', 'bunch', 'question', 'somebody', 'anybody', 'dirt', 'next', 'round', 'powerbook', 'introduction', 'expect', 'hear', 'suppose', 'appearence', 'summer', 'hear', 'access', 'macleak', 'wonder', 'anybody', 'info', 'anybody', 'hear', 'rumor', 'price', 'drop', 'powerbook', 'line', 'duo', 'impression', 'display', 'swing', 'disk', 'feel', 'good', 'di